# Gazebo-Sim 2D Registration Result

Shows the **last registered pair** saved by `viewGazeboSimPairs.py`
(`input1.csv`, `input2.csv`, `registration_meta.csv` in the viewer's
`PLOT_DATA_DIR`). No registration is run here.

**Prerequisites:**
- Run `viewGazeboSimPairs.py` once (same conda env) so the pair files exist
- Kernel working directory must be the `radarDataset` folder (VS Code default when opening this notebook)


In [1]:
import os
import sys

import numpy as np
import pandas as pd
import cv2

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reuse the viewer's PLOT_DATA_DIR and warp conventions (single source of truth)
sys.path.insert(0, os.getcwd())
import viewGazeboSimPairs as vp

DATA_DIR = vp.PLOT_DATA_DIR
print(f"Pair data directory: {DATA_DIR}")
print(f"Noise level in viewer config: {vp.NOISE_LEVEL}")

Pair data directory: /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data_gazebosim
Noise level in viewer config: high


In [2]:
def load_csv_array(filepath):
    if not os.path.exists(filepath):
        return None
    return np.loadtxt(filepath)


def load_meta_csv(filepath):
    if not os.path.exists(filepath):
        return None
    return pd.read_csv(filepath)


def _norm(a):
    return (a - a.min()) / (a.max() - a.min() + 1e-10)


img1 = load_csv_array(os.path.join(DATA_DIR, "input1.csv"))
img2 = load_csv_array(os.path.join(DATA_DIR, "input2.csv"))
meta_df = load_meta_csv(os.path.join(DATA_DIR, "registration_meta.csv"))

if img1 is None or img2 is None or meta_df is None or len(meta_df) == 0:
    print("ERROR: pair files missing or empty in", DATA_DIR)
    print("  input1.csv exists:          ", os.path.exists(os.path.join(DATA_DIR, "input1.csv")))
    print("  input2.csv exists:          ", os.path.exists(os.path.join(DATA_DIR, "input2.csv")))
    print("  registration_meta.csv exists:", os.path.exists(os.path.join(DATA_DIR, "registration_meta.csv")))
    print("Run viewGazeboSimPairs.py first, then re-run this notebook.")
else:
    meta = meta_df.iloc[-1]
    N = int(meta["N"])
    pixel_size = float(meta["pixel_size_m"])
    frame1, frame2 = int(meta["frame1"]), int(meta["frame2"])
    print(f"Pair: frames {frame1} -> {frame2}")
    print(f"Method: {meta['method']} | N={N} | radius={meta['radius_m']} m | pixel_size={pixel_size:.4f} m")
    print(f"Rot: {meta['rot_angle_deg']:.3f} deg | Tx: {meta['tx_m']:.3f} m | Ty: {meta['ty_m']:.3f} m")
    print(f"Confidence: {meta['confidence']:.3f} | Time: {meta['time_ms']:.1f} ms | Solutions: {int(meta['n_solutions'])}")
    print(f"GT error: rot={meta['gt_rot_err_deg']:.3f} deg | trans={meta['gt_trans_err_m']:.3f} m")
    print()
    print(meta_df.to_string(index=False))

Pair: frames 10 -> 11
Method: lightglue | N=256 | radius=10 m | pixel_size=0.0781 m
Rot: 20.267 deg | Tx: 0.483 m | Ty: 0.152 m
Confidence: 0.515 | Time: 1003.1 ms | Solutions: 0
GT error: rot=0.322 deg | trans=0.030 m

 frame1  frame2  rot_angle_deg     tx_m   ty_m  confidence    time_ms  gt_rot_err_deg  gt_trans_err_m   N  n_solutions  radius_m  pixel_size_m    method
     10      11       20.26727 0.483295 0.1523    0.515152 1003.09968         0.32152        0.029908 256            0        10      0.078125 lightglue


In [3]:
if img1 is not None and img2 is not None and meta_df is not None and len(meta_df) > 0:
    title = (f"Pair (frames {frame1}->{frame2})<br>"
             f"Rot: {meta['rot_angle_deg']:.2f} deg | Tx: {meta['tx_m']:.2f} m | Ty: {meta['ty_m']:.2f} m<br>"
             f"Confidence: {meta['confidence']:.3f} | Time: {meta['time_ms']:.1f} ms<br>"
             f"GT Error: Rot={meta['gt_rot_err_deg']:.2f} deg | Trans={meta['gt_trans_err_m']:.2f} m")

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f"Input {frame1}", f"Input {frame2}"),
                        specs=[[{"type": "heatmap"}, {"type": "heatmap"}]],
                        horizontal_spacing=0.1)
    fig.add_trace(go.Heatmap(z=_norm(img1), colorscale="Viridis", showscale=False), row=1, col=1)
    fig.add_trace(go.Heatmap(z=_norm(img2), colorscale="Viridis", showscale=False), row=1, col=2)
    fig.update_layout(height=500, width=1000, title_text=title, title_font_size=14, title_x=0.5)
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)
    fig.update_yaxes(scaleanchor="x2", scaleratio=1, row=1, col=2)
    fig.show()
else:
    print("Cannot display: missing data files.")

In [4]:
if img1 is not None and img2 is not None and meta_df is not None and len(meta_df) > 0:
    # Rebuild the standard SE3 transform from the saved (already fixed) values
    # and warp img2 exactly like viewGazeboSimPairs.run_pair() does.
    yaw = np.deg2rad(float(meta["rot_angle_deg"]))
    c, s = np.cos(yaw), np.sin(yaw)
    T = np.eye(4)
    T[:3, :3] = [[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]]
    T[:3, 3] = [float(meta["tx_m"]), float(meta["ty_m"]), 0.0]

    affine = vp.get_lidarsim_affine(T, pixel_size=pixel_size, img_size=N)
    warped = cv2.warpPerspective(img2, affine, (N, N))
    blended = 0.5 * img1 + 0.5 * warped

    fig2 = go.Figure(data=go.Heatmap(z=_norm(blended), colorscale="Viridis", showscale=True,
                                     colorbar_title="Intensity"))
    fig2.update_layout(title_text=f"Blended (warped overlay) - Pair {frame1}->{frame2}",
                       title_x=0.5, height=700, width=700)
    fig2.update_xaxes(showticklabels=False, showgrid=False)
    fig2.update_yaxes(showticklabels=False, showgrid=False)
    fig2.show()
else:
    print("Cannot display: missing data files.")